# Passo 1: Monta Google Drive

In [13]:
from google.colab import drive
from google.colab import userdata

In [14]:
drive.mount('/content/drive')

# NOTA: Dopo aver eseguito questa cella, dovrai cliccare su un link
# autorizzare il tuo account Google e incollare il codice di autorizzazione.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Passo 2: Naviga nella Cartella del Progetto

In [15]:
import os
# ATTENZIONE: Modifica questo percorso perché corrisponda a dove hai salvato
# la cartella del tuo progetto su Google Drive.
PROJECT_PATH = '/content/drive/MyDrive/basketball_analysis/basketball_analysis'

os.chdir(PROJECT_PATH)

# Verifica di essere nella cartella giusta listando i file
!ls

ball_acquisition		 shot_detector
basketball-court-detection-2-1	 shot_visualizer
basketball-player-detection-2-1  speed_and_distance_calculator
configs				 stubs
court_keypoint_detector		 tactical_view
Dockerfile			 tactical_view_converter
drawers				 team_assigner
extract_frames.py		 test_ball_drawer_output.jpg
GEMINI.md			 test_ball_drawer.py
generate_stubs.py		 test_court_drawer_output.jpg
images				 test_court_drawer.py
input_videos			 test_player_drawer_output.jpg
main.py				 test_player_drawer.py
models				 test_shot_visualizer_output.jpg
output_videos			 test_shot_visualizer.py
pass_and_interception_detector	 trackers
README.md			 training_notebooks
requirements.txt		 utils
roboflow_dataset		 yolo11n.pt
runs				 yolov8l.pt
shot_classifier


Dovresti vedere l'elenco dei file del tuo progetto (`main.py`, `models/`, `input_videos/`, ecc.).


# Passo 3: Installa le Librerie Necessarie

In [16]:
# Installa le versioni più recenti di Ultralytics e Roboflow
!pip install ultralytics roboflow -q

L'opzione `-q` (quiet) riduce l'output, mantenendo il notebook più pulito.

# Cella 4: Download del Dataset

In [ ]:
# Incolla qui lo snippet di codice che hai copiato da Roboflow.
# Dovrebbe essere simile a questo, ma con i tuoi dati specifici.
# ASSICURATI DI USARE LO SNIPPET DEL NUOVO DATASET (quello che abbiamo appena creato)

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')

from roboflow import Roboflow
rf = Roboflow(api_key="ROBOFLOW_API_KEY")
project = rf.workspace("firstworkspace-riuxx").project("basketball-player-detection-2-68vvt")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


Importante: Assicurati di usare lo snippet corretto del progetto "basketball-court-detection-2" e della versione che hai appena generato.


# Come verificare che il modello sia davvero addestrato sul tuo dataset

In [18]:
from ultralytics import YOLO

m = YOLO("runs/detect/unified_detector_v2/weights/best.pt")  # metti il tuo path
print("Classi (names):", m.names)           # devono essere le tue (palla, canestro, ecc.)
print("Numero classi (nc):", m.model.nc)    # deve combaciare con il tuo data.yaml

# Confronto veloce con il base
base = YOLO("yolov8l.pt")
print("Stesse classi del base?", m.names == base.names)  # deve risultare False


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Classi (names): {0: 'ball', 1: 'number', 2: 'player', 3: 'player-dribble', 4: 'player-fall', 5: 'player-jump-shot', 6: 'player-layup', 7: 'player-screen', 8: 'player-shot-block', 9: 'referee', 10: 'rim'}
Numero classi (nc): 11
Stesse classi del base? False


In [19]:
# Se hai il run completo
!cat runs/detect/unified_detector_v2/results.csv | tail -n 1
# oppure
from ultralytics.utils import LOGGER
print(m.metrics)  # se presente


116,13313.6,0.47075,0.25599,0.84731,0.71485,0.69755,0.70264,0.53039,0.72109,0.44132,0.98716,0.000160747,0.000160747,0.000160747
None


In [20]:
pred = m("runs/BasketGame_04_frame_33.jpg", save=True, conf=0.25)


image 1/1 /content/drive/MyDrive/basketball_analysis/basketball_analysis/runs/BasketGame_04_frame_33.jpg: 384x640 1 ball, 1 player, 1 rim, 116.5ms
Speed: 7.8ms preprocess, 116.5ms inference, 411.0ms postprocess per image at shape (1, 3, 384, 640)
Results saved to runs/detect/predict2


# Passo 5: Esegui il Fine-Tuning (La Cella Cruciale)

Questa è la cella che sostituisce il tuo vecchio comando `!yolo`. Useremo le API Python.

In [ ]:
from ultralytics import YOLO
# 1. Carica un modello YOLOv8 pre-addestrato per il RILEVAMENTO DI OGGETTI.
#    NON usiamo il tuo 'court_keypoint_detector' perché quello è specializzato
#    in un task diverso (pose/keypoint). Partire da un modello pre-addestrato
#    per il rilevamento di oggetti (come 'yolov8l.pt') è la scelta corretta
#    e darà risultati molto migliori. 'l' sta per Large, un modello potente.

print("Caricamento del modello YOLOv8 Large pre-addestrato...")
model = YOLO('yolov8l.pt') # pesi COCO pre-addestrati (consigliato)

# 2. Avvia il processo di fine-tuning sul nostro dataset unificato.
print(f"Inizio del training sul dataset che si trova in: {dataset.location}")
results = model.train(
    data=f'{dataset.location}/data.yaml',  # Percorso al file di configurazione del dataset
    epochs=150,                            # 150 epoche è un ottimo punto di partenza per il fine-tuning
    imgsz=640,                            # Dimensione delle immagini (deve corrispondere a Roboflow)
    project='runs/detect',                 # Cartella specifica per i risultati di 'detection'
    name='unified_detector_v2',            # Nome descrittivo per questo esperimento
    patience=30,                           # Si ferma automaticamente se non migliora per 50 epoche
    batch=8                                # Riduciamo il batch size per sicurezza con la memoria della GPU
)

print("Training completato!")

Caricamento del modello YOLOv8 Large pre-addestrato...
Inizio del training sul dataset che si trova in: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1
New https://pypi.org/project/ultralytics/8.3.182 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.179 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7

train: Scanning /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/train/labels.cache... 3831 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3831/3831 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 31.4±19.3 MB/s, size: 72.6 KB)


val: Scanning /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/valid/labels.cache... 253 images, 0 backgrounds, 0 corrupt: 100%|██████████| 253/253 [00:00<?, ?it/s]


Plotting labels to runs/detect/unified_detector_v2/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000667, momentum=0.9) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs/detect/unified_detector_v2
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

    116/150      6.73G     0.4707      0.256     0.8473        132        640: 100%|██████████| 479/479 [01:50<00:00,  4.34it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:03<00:00,  4.83it/s]

                   all        253       4212      0.715      0.698      0.703       0.53
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 86, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



116 epochs completed in 3.699 hours.
Optimizer stripped from runs/detect/unified_detector_v2/weights/last.pt, 87.7MB
Optimizer stripped from runs/detect/unified_detector_v2/weights/best.pt, 87.7MB

Validating runs/detect/unified_detector_v2/weights/best.pt...
Ultralytics 8.3.179 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
Model summary (fused): 112 layers, 43,615,089 parameters, 0 gradients, 164.9 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:04<00:00,  3.23it/s]


                   all        253       4212      0.748      0.684      0.714      0.538
                  ball        217        220      0.841      0.827      0.875      0.524
                number        235        940      0.792      0.809      0.857       0.52
                player        250       2028      0.924      0.973      0.968       0.83
        player-dribble        123        123      0.731      0.593      0.661      0.503
           player-fall         14         14      0.711      0.571      0.587      0.482
      player-jump-shot         27         27      0.582      0.704       0.78      0.655
          player-layup         13         13       0.39      0.297      0.231      0.131
         player-screen         12         12      0.591      0.417      0.476      0.364
     player-shot-block         27         29       0.73      0.373      0.444      0.333
               referee        246        581      0.959      0.974      0.985      0.864
                   ri



Spiegazione delle differenze chiave:

   * `model = YOLO(...)`: Invece di specificare il modello nella riga di comando, lo carichiamo prima come oggetto Python. Questo è il punto in cui carichiamo
     court_keypoint_detector_model_v2.pt.
   * `model.train(...)`: Chiamiamo il metodo .train() sull'oggetto model. Questo ci dà un controllo molto più pulito sui parametri.
   * `data=f'{dataset.location}/data.yaml'`: Usiamo una f-string per passare dinamicamente la posizione del file data.yaml che Roboflow ci ha fornito. È più robusto che
     scriverlo a mano.
   * `epochs=100`: Ho impostato 100 epoche. È un numero solido per il fine-tuning. Il tuo precedente epochs=500 era probabilmente eccessivo e avrebbe rischiato l'overfitting.
     YOLO ha un meccanismo di "early stopping", quindi si fermerà da solo se non vede miglioramenti per un certo numero di epoche (di default 50).
   * `project` e `name`: Questi parametri ci permettono di organizzare meglio i risultati. Il tuo nuovo modello e i grafici di training verranno salvati in
     runs/pose/finetune_court_detector/.

# DOPO L'ADDESTRAMENTO:

VERIFICA 1

In [ ]:
START = # TODO: modifica con path modello - dopo addestramento

import torch, os
import ultralytics
print("torch:", torch.__version__, "| ultralytics:", ultralytics.__version__)

ckpt = torch.load(START, map_location="cpu", weights_only=False)  # <-- chiave
print("Chiavi:", list(ckpt.keys())[:10])
print("Optimizer presente?", ("optimizer" in ckpt) and (ckpt["optimizer"] is not None))
print("Epoch salvata:", ckpt.get("epoch"))
print("Versione Ultralytics nel ckpt:", ckpt.get("version"))
print("Dimensione file (MB):", os.path.getsize(START)/1024/1024)

VERIFICA 2

In [ ]:
import torch, os
ckpt = torch.load(START, map_location="cpu", weights_only=False)
print("optimizer?", ckpt.get("optimizer") is not None, "| epoch:", ckpt.get("epoch"))
print("sizeMB:", os.path.getsize(START)/1024/1024)

VERIFICA 3

In [ ]:
import os, torch
ckpt = torch.load(START, map_location="cpu", weights_only=False)
data_yaml_saved = ckpt.get("train_args", {}).get("data")
print("data.yaml salvato:", data_yaml_saved, "| esiste?", os.path.exists(str(data_yaml_saved)))

# DOWNLOAD FINALE - SE NECESSARIO - DOVREBBE SALVARE IN AUTOMATICO SU DRIVE

In [ ]:
from google.colab import files

# Il percorso corretto ora riflette i parametri 'project' e 'name' della cella di training
model_path = 'runs/detect/unified_detector_v1.1/weights/best.pt'
print(f"Download del modello migliore da: {model_path}")
files.download(model_path)

Download del modello migliore da: runs/detect/unified_detector_v1.1/weights/best.pt


FileNotFoundError: Cannot find file: runs/detect/unified_detector_v1.1/weights/best.pt

# Passo 6: Verifica i Risultati

Dopo che la Cella 5 ha finito, il tuo nuovo modello sarà stato salvato. Puoi verificare il percorso esatto e usarlo. Il file che ti interessa sarà in:
  `runs/pose/finetune_court_detector/weights/best.pt`

Questo è il file che dovrai scaricare da Google Drive e usare nella tua applicazione principale.

Questo nuovo notebook è più pulito, più robusto e, soprattutto, esegue correttamente il fine-tuning come avevamo pianificato.